# Figure 7: reverse-drug query

Chemical candidate ranking for a stronger hypoxia response in endometrial tumors and the reverse-drug experimental-rank panel.

In [ ]:

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image

TASK_ROOT = Path('/Users/dudu/Documents/3_Project/12_PxFquery/5_phase_translation/G-011_paper_figure_planning/T-150_figure7_reverse_drug_query_notebook_figure_workb')
ROUND_ID = 'SCAN02_PXFQUERY_TEST'
TABLE_DIR = TASK_ROOT / '4_artifact/5_table/figure7/figure7a_clean_rerun' / ROUND_ID
RUN_DIR = TASK_ROOT / '4_artifact/1_tmp/figure7a_clean_rerun_qdata_answers' / ROUND_ID
PIC_DIR = TASK_ROOT / '4_artifact/4_picture/figure7/figure7a_clean_rerun' / ROUND_ID
summary = pd.read_csv(TABLE_DIR / 'scan02_pxfquery_test_summary_full.csv')
ranked = pd.read_csv(TABLE_DIR / 'scan02_pxfquery_test_ranked_top20.csv')
route_functions = pd.read_csv(TABLE_DIR / 'scan02_pxfquery_test_route_target_functions.csv')
routes = pd.read_csv(TABLE_DIR / 'scan02_pxfquery_test_route_summary.csv')
fig_specs = pd.read_csv(TABLE_DIR / 'scan02_pxfquery_test_figure_specs.csv')


In [ ]:

QUERIES = [
    ('scan02_q01_endometrial_dna_damage', 'Which small molecules could make endometrial tumors show a stronger response to DNA damage?'),
    ('scan02_q02_endometrial_immune_alarm', 'Which small molecules could make endometrial tumors send stronger immune alarm signals?'),
    ('scan02_q03_kidney_immune_alarm', 'Which small molecules could make kidney tumors send stronger immune alarm signals?'),
    ('scan02_q04_kidney_dna_damage', 'Which small molecules could make kidney tumors show a stronger response to DNA damage?'),
]
commands = []
runner = TASK_ROOT / '3_execution/run_figure7a_single_reverse_drug_query.py'
context = {
    'scan02_q01_endometrial_dna_damage': 'endometrial cancer',
    'scan02_q02_endometrial_immune_alarm': 'endometrial cancer',
    'scan02_q03_kidney_immune_alarm': 'kidney cancer',
    'scan02_q04_kidney_dna_damage': 'kidney cancer',
}
for run_id, query in QUERIES:
    commands.append(f'python {runner} --round-id {ROUND_ID} --run-id {run_id} --source-context "{context[run_id]}" --top-n 20 --query "{query}"')
print('\n'.join(commands))


In [ ]:

display(summary[['run_id','status','source_context','query','interpreted_question','summary']])


In [ ]:

top_cols = ['run_id','rank','label','score','support_routes','support_cells','cells','pert_id','cmap_name']
display(ranked[ranked['rank'] <= 8][top_cols])


In [ ]:

route_cols = ['run_id','route_id','cell','modality','status','cell_match_type','route_quality','n_rows','n_ranked_groups']
display(routes[route_cols])


In [ ]:

func_cols = ['run_id','route_id','cell','function_rank','label','var_name','direction','input','route_quality','cell_match_type']
display(route_functions[func_cols].head(80))


In [ ]:

for _, row in summary.iterrows():
    display(Markdown(f"### {row['run_id']}"))
    display(Markdown(f"**Query:** {row['query']}"))
    if isinstance(row.get('route_png'), str) and row['route_png']:
        first = row['route_png'].split('; ')[0]
        display(Image(filename=first, width=900))
    display(Markdown(f"PDF: `{row.get('route_pdf','')}`"))


## Reverse-drug benchmark panel

In [ ]:
from __future__ import annotations

import argparse
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


PXF_COLOR = "#d62728"
LLM_COLOR = "#1f77b4"
SYSTEMS = ["PxFquery", "Direct LLM"]

FORWARD_PANELS = [
    ("activated_top3", "Activated top3"),
    ("suppressed_top3", "Suppressed top3"),
    ("activated_top5", "Activated top5"),
    ("suppressed_top5", "Suppressed top5"),
]
REVERSE_PANELS = [
    ("recommendations_top1", "Top1"),
    ("recommendations_top3", "Top3"),
    ("recommendations_top5", "Top5"),
    ("recommendations_top10", "Top10"),
]


@dataclass(frozen=True)
class DatasetSpec:
    figure_id: str
    task_type: str
    source_name: str
    output_stem: str
    panels: list[tuple[str, str]]


DATASETS = [
    DatasetSpec(
        figure_id="fig4_panel_b",
        task_type="forward_genetic",
        source_name="forward_genetic_xpr100_first100",
        output_stem="fig4_panel_b_forward_genetic_xpr",
        panels=FORWARD_PANELS,
    ),
    DatasetSpec(
        figure_id="fig5_panel_b",
        task_type="forward_drug",
        source_name="forward_drug_100_first100",
        output_stem="fig5_panel_b_forward_drug",
        panels=FORWARD_PANELS,
    ),
    DatasetSpec(
        figure_id="fig6_panel_b",
        task_type="reverse_genetic",
        source_name="reverse_genetic_100",
        output_stem="fig6_panel_b_reverse_genetic",
        panels=REVERSE_PANELS,
    ),
    DatasetSpec(
        figure_id="fig7_panel_b",
        task_type="reverse_drug",
        source_name="reverse_drug_100",
        output_stem="fig7_panel_b_reverse_drug",
        panels=REVERSE_PANELS,
    ),
]


def configure_matplotlib() -> None:
    mpl.rcParams.update(
        {
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "font.family": "Arial",
            "axes.unicode_minus": False,
        }
    )


def query_order(query_id: str) -> int:
    match = re.search(r"_(\d+)$", str(query_id))
    return int(match.group(1)) if match else 10**9


def pvalue_stars(pvalue: float | None) -> str:
    if pvalue is None or not np.isfinite(pvalue):
        return "n.s."
    if pvalue < 1e-4:
        return "****"
    if pvalue < 1e-3:
        return "***"
    if pvalue < 1e-2:
        return "**"
    if pvalue < 0.05:
        return "*"
    return "n.s."


def paired_pvalue(data: pd.DataFrame) -> tuple[float | None, int]:
    pivot = data.pivot_table(
        index="query_id",
        columns="system",
        values="median_rank_percentile",
        aggfunc="first",
    ).dropna(subset=SYSTEMS)
    if len(pivot) < 2:
        return None, len(pivot)
    diff = pivot["PxFquery"].astype(float) - pivot["Direct LLM"].astype(float)
    if np.allclose(diff.to_numpy(), 0):
        return 1.0, len(pivot)
    return float(wilcoxon(pivot["PxFquery"], pivot["Direct LLM"]).pvalue), len(pivot)


def first100_canonical_ids(answers_path: Path) -> list[str]:
    ids: list[str] = []
    with answers_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            gate = record.get("evidence_resolution_gate") or {}
            rows = ((record.get("answer_tables") or {}).get("ranked_results") or [])
            if gate.get("answer_unresolved") is False and rows:
                ids.append(str(record.get("query_id")))
    return sorted(ids, key=query_order)[:100]


def build_forward_query_median(checks_path: Path, query_ids: Iterable[str]) -> pd.DataFrame:
    selected_ids = {str(query_id) for query_id in query_ids}
    checks = pd.read_csv(checks_path)
    checks = checks[checks["query_id"].astype(str).isin(selected_ids)].copy()
    checks = checks[checks["not_found"].astype(str).str.lower() != "true"].copy()
    checks["rank_percentile"] = pd.to_numeric(checks["rank_percentile"], errors="coerce")
    return (
        checks.dropna(subset=["rank_percentile"])
        .groupby(["query_id", "task_type", "field", "system"], dropna=False)
        .agg(
            median_rank_percentile=("rank_percentile", "median"),
            mean_rank_percentile=("rank_percentile", "mean"),
            n_found=("rank_percentile", "size"),
            same_direction_count=("same_direction", lambda values: int(values.astype(str).str.lower().eq("true").sum())),
        )
        .reset_index()
    )


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def prepare_release_data(task_root: Path, release_root: Path) -> None:
    table_root = task_root / "4_artifact/5_table"
    data_dir = release_root / "data"

    forward_sources = [
        (
            "forward_genetic_xpr100_first100",
            "fg_xpr100_resolved_official_t144_20260703",
        ),
        (
            "forward_drug_100_first100",
            "fd100_resolved_official_t144_20260703",
        ),
    ]
    for output_name, source_dir_name in forward_sources:
        source_dir = table_root / source_dir_name
        query_ids = first100_canonical_ids(source_dir / "filtered_batch_pxfquery_answers.jsonl")
        query_median = build_forward_query_median(source_dir / "filtered_batch_checks.csv", query_ids)
        write_csv(pd.DataFrame({"query_id": query_ids}), data_dir / "query_ids" / f"{output_name}_query_ids.csv")
        write_csv(query_median, data_dir / "query_medians" / f"{output_name}_query_median.csv")

    reverse = pd.read_csv(table_root / "rg_rd_batch100_current/rg_rd_batch100_current_query_median.csv")
    for task_type, output_name in [
        ("reverse_genetic", "reverse_genetic_100"),
        ("reverse_drug", "reverse_drug_100"),
    ]:
        query_median = reverse[reverse["task_type"] == task_type].copy()
        query_ids = sorted(query_median["query_id"].astype(str).unique(), key=query_order)
        write_csv(pd.DataFrame({"query_id": query_ids}), data_dir / "query_ids" / f"{output_name}_query_ids.csv")
        write_csv(query_median, data_dir / "query_medians" / f"{output_name}_query_median.csv")

    source_manifest = pd.DataFrame(
        [
            {
                "dataset": "forward_genetic_xpr100_first100",
                "source": str(table_root / "fg_xpr100_resolved_official_t144_20260703"),
                "selection": "first 100 canonical evaluable xpr forward genetic queries with nonempty ranked evidence table",
            },
            {
                "dataset": "forward_drug_100_first100",
                "source": str(table_root / "fd100_resolved_official_t144_20260703"),
                "selection": "first 100 canonical evaluable forward drug queries with nonempty ranked evidence table",
            },
            {
                "dataset": "reverse_genetic_100",
                "source": str(table_root / "rg_rd_batch100_current/rg_rd_batch100_current_query_median.csv"),
                "selection": "existing reverse genetic benchmark query-median table",
            },
            {
                "dataset": "reverse_drug_100",
                "source": str(table_root / "rg_rd_batch100_current/rg_rd_batch100_current_query_median.csv"),
                "selection": "existing reverse drug benchmark query-median table",
            },
        ]
    )
    write_csv(source_manifest, data_dir / "audit" / "source_manifest.csv")


def draw_panel(ax: plt.Axes, data: pd.DataFrame, title: str) -> dict[str, object]:
    values_by_system = [
        data[data["system"] == system]["median_rank_percentile"].dropna().astype(float).to_numpy()
        for system in SYSTEMS
    ]
    violin = ax.violinplot(
        values_by_system,
        positions=[0, 1],
        widths=0.72,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, color in zip(violin["bodies"], [PXF_COLOR, LLM_COLOR]):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.16)
        body.set_linewidth(0.9)

    for x, vals in enumerate(values_by_system):
        if len(vals):
            q1, median, q3 = np.percentile(vals, [25, 50, 75])
            ax.vlines(x, q1, q3, color="#555555", linewidth=2.2, alpha=0.58, zorder=3)
            ax.hlines(q1, x - 0.13, x + 0.13, color="#555555", linewidth=1.1, alpha=0.58, zorder=3)
            ax.hlines(q3, x - 0.13, x + 0.13, color="#555555", linewidth=1.1, alpha=0.58, zorder=3)
            ax.hlines(median, x - 0.18, x + 0.18, color="black", linewidth=1.25, zorder=4)

    rng = np.random.default_rng(20260702)
    for x, color, vals in zip([0, 1], [PXF_COLOR, LLM_COLOR], values_by_system):
        jitter = rng.uniform(-0.13, 0.13, len(vals))
        ax.scatter(
            np.full(len(vals), x) + jitter,
            vals,
            s=16,
            c=color,
            edgecolor="white",
            linewidth=0.22,
            alpha=0.36,
            zorder=5,
        )

    pvalue, paired_n = paired_pvalue(data)
    significance = pvalue_stars(pvalue)
    ax.text(0.5, 0.985, significance, transform=ax.transAxes, ha="center", va="top", fontsize=10)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(SYSTEMS, fontsize=8)
    ax.set_ylim(105, -5)
    ax.grid(axis="y", alpha=0.22)
    return {
        "paired_n": paired_n,
        "paired_wilcoxon_p": pvalue,
        "significance": significance,
        "pxf_median": float(np.nanmedian(values_by_system[0])) if len(values_by_system[0]) else np.nan,
        "llm_median": float(np.nanmedian(values_by_system[1])) if len(values_by_system[1]) else np.nan,
    }


def plot_dataset(release_root: Path, spec: DatasetSpec) -> pd.DataFrame:
    query_median_path = release_root / "data/query_medians" / f"{spec.source_name}_query_median.csv"
    data = pd.read_csv(query_median_path)
    data = data[data["task_type"] == spec.task_type].copy()
    data["median_rank_percentile"] = pd.to_numeric(data["median_rank_percentile"], errors="coerce")

    fig, axes = plt.subplots(2, 2, figsize=(6.6, 5.8), sharey=True)
    summary_rows = []
    for ax, (field, title) in zip(axes.ravel(), spec.panels):
        panel_data = data[data["field"] == field].copy()
        row = draw_panel(ax, panel_data, title)
        row.update(
            {
                "figure_id": spec.figure_id,
                "task_type": spec.task_type,
                "field": field,
                "field_title": title,
            }
        )
        summary_rows.append(row)
    for ax in axes.ravel()[::2]:
        ax.set_ylabel("Experimental rank percentile (%)")
    fig.tight_layout()

    pdf_path = release_root / "figures/pdf" / f"{spec.output_stem}.pdf"
    png_path = release_root / "figures/png" / f"{spec.output_stem}.png"
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    png_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return pd.DataFrame(summary_rows)


def plot_release_figures(release_root: Path) -> pd.DataFrame:
    configure_matplotlib()
    summaries = [plot_dataset(release_root, spec) for spec in DATASETS]
    summary = pd.concat(summaries, ignore_index=True)
    write_csv(summary, release_root / "data/audit/panel_b_summary.csv")

    figure_manifest = pd.DataFrame(
        [
            {
                "figure_id": spec.figure_id,
                "task_type": spec.task_type,
                "source_name": spec.source_name,
                "pdf": str(Path("figures/pdf") / f"{spec.output_stem}.pdf"),
                "png": str(Path("figures/png") / f"{spec.output_stem}.png"),
            }
            for spec in DATASETS
        ]
    )
    write_csv(figure_manifest, release_root / "data/audit/figure_manifest.csv")
    return summary


def parse_args() -> argparse.Namespace:
    default_release_root = Path(__file__).resolve().parents[1]
    default_task_root = default_release_root.parents[2]
    parser = argparse.ArgumentParser(description="Prepare and plot PxFquery panel B benchmark figures.")
    parser.add_argument("--task-root", type=Path, default=default_task_root)
    parser.add_argument("--release-root", type=Path, default=default_release_root)
    parser.add_argument("--mode", choices=["all", "prepare", "plot"], default="all")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.mode in {"all", "prepare"}:
        prepare_release_data(args.task_root, args.release_root)
    if args.mode in {"all", "plot"}:
        summary = plot_release_figures(args.release_root)
        print(summary.to_string(index=False))


if __name__ == "__main__":
    main()


## Evidence network graph\n\nThe original rendering code retained for this figure.

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import pickle
import sys
import time
import traceback
from pathlib import Path


PROJECT_ROOT = Path("/Users/dudu/Documents/3_Project/12_PxFquery")
TASK_ROOT = PROJECT_ROOT / "5_phase_translation/G-011_paper_figure_planning/T-150_figure7_reverse_drug_query_notebook_figure_workb"
T144_ROOT = PROJECT_ROOT / "4_phase_development/G-035_goal_ms8_human_usable_package/T-144_ms8_14_user_value_l5_output_rework"
SRC_ROOT = TASK_ROOT / "3_execution/t144_package_source_for_fig7a/src"
RESOURCE_ROOT = Path("/Users/dudu/.cache/pxfquery/resources/v20260628")

sys.path.insert(0, str(SRC_ROOT))
os.environ["PYTHONPATH"] = str(SRC_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")


def load_env() -> None:
    for env_path in [Path.home() / ".env", TASK_ROOT / "1_asset/.env", T144_ROOT / "1_asset/.env"]:
        if not env_path.exists():
            continue
        for raw in env_path.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())
    if os.environ.get("DEEPSEEK_API_KEY") and not os.environ.get("PXFQUERY_LLM_API_KEY"):
        os.environ["PXFQUERY_LLM_API_KEY"] = os.environ["DEEPSEEK_API_KEY"]
    if os.environ.get("DEEPSEEK_API_BASE") and not os.environ.get("PXFQUERY_LLM_BASE_URL"):
        os.environ["PXFQUERY_LLM_BASE_URL"] = os.environ["DEEPSEEK_API_BASE"]
    os.environ.setdefault("PXFQUERY_LLM_PROVIDER", "deepseek")
    os.environ.setdefault("PXFQUERY_LLM_MODEL", "deepseek-v4-flash")


def answer_to_dict(answer):
    if hasattr(answer, "to_dict"):
        return answer.to_dict()
    if hasattr(answer, "__dict__"):
        return dict(answer.__dict__)
    return {"text": str(answer)}


def get_summary(answer_dict: dict) -> str:
    for key in ["summary", "biological_summary", "answer", "text"]:
        value = answer_dict.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    for value in answer_dict.values():
        if isinstance(value, str) and len(value) > 40:
            return value.strip()
    return str(answer_dict)[:1000]


def load_brd_name_index() -> dict[str, str]:
    drug_index_path = RESOURCE_ROOT / "drug_index.json"
    if not drug_index_path.exists():
        return {}
    try:
        drug_index = json.loads(drug_index_path.read_text(encoding="utf-8"))
    except Exception:
        return {}
    out: dict[str, str] = {}
    for name, value in drug_index.items():
        values = value if isinstance(value, list) else [value]
        for brd in values:
            if isinstance(brd, str):
                out.setdefault(brd, str(name))
    return out


BRD_NAME_INDEX = load_brd_name_index()


def display_label(row: dict) -> str:
    label = row.get("label") or row.get("cmap_name") or row.get("pert_id")
    pert_id = row.get("pert_id") or row.get("cmap_name")
    if (not label or label == "Unnamed compound") and isinstance(pert_id, str):
        return BRD_NAME_INDEX.get(pert_id, pert_id)
    if isinstance(label, str) and label.startswith("BRD-"):
        return BRD_NAME_INDEX.get(label, label)
    return str(label)


def compact_top(answer_dict: dict) -> list[dict]:
    ranked = ((answer_dict.get("tables") or {}).get("ranked_results") or [])[:8]
    rows = []
    for index, row in enumerate(ranked, start=1):
        rows.append(
            {
                "rank": row.get("rank", index),
                "label": display_label(row),
                "raw_label": row.get("label"),
                "cmap_name": row.get("cmap_name"),
                "pert_id": row.get("pert_id"),
                "score": row.get("score"),
                "support": row.get("support") or row.get("support_count") or row.get("match_support_count"),
                "cells": row.get("cells") or row.get("cell_iname") or row.get("support_cells"),
            }
        )
    return rows


def compact_targets(answer_dict: dict) -> list[dict]:
    targets = ((answer_dict.get("tables") or {}).get("route_target_functions") or [])[:20]
    return [
        {
            "route_id": row.get("route_id"),
            "label": row.get("label") or row.get("function_label") or row.get("var_name"),
            "direction": row.get("direction"),
            "source": row.get("source"),
            "cell": row.get("cell") or row.get("cell_iname"),
        }
        for row in targets
    ]


def compact_routes(answer_dict: dict) -> list[dict]:
    routes = ((answer_dict.get("tables") or {}).get("route_summary") or [])[:20]
    keys = ["route_id", "status", "cell", "cell_iname", "modality", "route_quality", "route_quality_score"]
    return [{key: row.get(key) for key in keys if key in row} for row in routes]


def compact_fig_specs(answer) -> list[dict]:
    specs = []
    for spec in getattr(answer, "figures", []) or []:
        item = {"kind": spec.get("kind"), "title": spec.get("title")}
        if spec.get("kind") == "reverse_layered_route_graph":
            item.update(
                {
                    "target_functions": len(spec.get("target_functions", [])),
                    "candidates": len(spec.get("candidates", [])),
                }
            )
        elif spec.get("kind") == "reverse_candidate_bubble":
            item.update({"candidates": len(spec.get("candidates", [])), "top": spec.get("candidates", [])[:5]})
        elif spec.get("kind") == "reverse_function_ring_heatmap":
            item.update(
                {
                    "source": spec.get("source"),
                    "functions": len(spec.get("functions", [])),
                    "matches": len(spec.get("matches", [])),
                }
            )
        specs.append(item)
    return specs


def run_one(args: argparse.Namespace) -> dict:
    load_env()
    from pxfquery import PxFQuery

    fig_root = TASK_ROOT / "4_artifact/4_picture/figure7/figure7a_clean_rerun" / args.round_id / args.run_id
    run_root = TASK_ROOT / "4_artifact/1_tmp/figure7a_clean_rerun_qdata_answers" / args.round_id / args.run_id
    table_root = TASK_ROOT / "4_artifact/5_table/figure7/figure7a_clean_rerun" / args.round_id
    for path in [fig_root / "png", fig_root / "pdf", run_root, table_root]:
        path.mkdir(parents=True, exist_ok=True)

    started = time.perf_counter()
    result = {
        "round_id": args.round_id,
        "run_id": args.run_id,
        "source_context": args.source_context,
        "query": args.query,
        "status": "started",
    }
    try:
        client = PxFQuery()
        qdata = client.tl.parse(args.query, top_n=args.top_n, progress=False)
        client.tl.answer(qdata, progress=False)
        answer = client.get.answer(qdata)
        answer_dict = answer_to_dict(answer)
        with open(run_root / f"{args.run_id}_qdata.pkl", "wb") as handle:
            pickle.dump(qdata, handle)
        answer_path = run_root / f"{args.run_id}_answer.json"
        answer_path.write_text(json.dumps(answer_dict, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        client.tl.figures(qdata, output_dir=fig_root / "pdf", prefix=args.run_id)
        pdf_paths = [str(Path(path)) for path in qdata.uns.get("figure_outputs", [])]
        client.tl.figures(qdata, output_dir=fig_root / "png", prefix=args.run_id, format="png")
        png_paths = [str(Path(path)) for path in qdata.uns.get("figure_outputs", [])]
        route_png = [path for path in png_paths if "route" in Path(path).name or "evidence" in Path(path).name or "layered" in Path(path).name]
        route_pdf = [path for path in pdf_paths if "route" in Path(path).name or "evidence" in Path(path).name or "layered" in Path(path).name]
        result.update(
            {
                "status": "ok",
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "answer_summary": get_summary(answer_dict),
                "top_candidates": compact_top(answer_dict),
                "target_functions": compact_targets(answer_dict),
                "routes": compact_routes(answer_dict),
                "figure_specs": compact_fig_specs(answer),
                "png_paths": png_paths,
                "pdf_paths": pdf_paths,
                "route_png_paths": route_png,
                "route_pdf_paths": route_pdf,
                "answer_json_path": str(answer_path),
                "qdata_pickle_path": str(run_root / f"{args.run_id}_qdata.pkl"),
            }
        )
    except Exception as exc:
        result.update(
            {
                "status": "error",
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(),
            }
        )
    out_path = table_root / f"{args.run_id}_result.json"
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(json.dumps({"run_id": args.run_id, "status": result["status"], "elapsed_seconds": result.get("elapsed_seconds")}, ensure_ascii=False))
    return result


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--round-id", required=True)
    parser.add_argument("--run-id", required=True)
    parser.add_argument("--source-context", required=True)
    parser.add_argument("--query", required=True)
    parser.add_argument("--top-n", type=int, default=8)
    run_one(parser.parse_args())


if __name__ == "__main__":
    main()


## 100-query reverse-drug benchmark

This notebook retains the Figure 7 query set, the reverse-drug candidate-selection rule, and the query-level benchmark inputs. `benchmark_data/` contains only the reverse-drug records used for this panel.


In [ ]:
from pathlib import Path
import pandas as pd

benchmark_dir = Path("benchmark_data")
query_set = pd.read_csv(benchmark_dir / "query_set.csv")
query_ids = pd.read_csv(benchmark_dir / "query_ids.csv")
query_median = pd.read_csv(benchmark_dir / "query_median.csv")

query_set.head(), query_ids.shape, query_median.groupby(["field", "system"]).query_id.nunique()


In [ ]:
from run_reverse_drug_benchmark import (
    build_query_medians,
    question,
    select_candidate_rows,
)

# Candidate selection and query-level median aggregation are retained in the
# adjacent production script. The loaded files are its final Figure 7 inputs.
